# Practical 09: Missing Value Imputation with SimpleImputer & Categorical Encoding

**UNIT 3: Preparing Big Data for Machine Learning**

**Problem Statement:** Write a Python program using Scikit-Learn to preprocess a machine learning dataset: handle missing numerical values using SimpleImputer with the mean strategy, transform categorical text labels into ordinal integers using LabelEncoder, and convert nominal variables to binary dummy indicators using One-Hot Encoding.

In [1]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

rs = np.random.default_rng(11)
n = 60
age = rs.integers(21, 59, n).astype(float)
salary = np.round(25500 + age * 890 + rs.normal(0, 6000, n), -2)
data = pd.DataFrame({
    "Country": rs.choice(["India", "Japan", "Brazil"], n),
    "Age": age,
    "Salary": salary,
    "Purchased": np.where(salary + rs.normal(0, 8000, n) > 56000, "Yes", "No"),
})
data.loc[rs.choice(n, 6, replace=False), "Age"] = np.nan
data.loc[rs.choice(n, 6, replace=False), "Salary"] = np.nan
print(data.head(9))

print(data.isna().sum())

imp = SimpleImputer(missing_values=np.nan, strategy="mean")
data[["Age", "Salary"]] = imp.fit_transform(data[["Age", "Salary"]])

print("Learned means (Age, Salary):", imp.statistics_.round(2))
print("\nMissing values after imputation:")
print(data.isna().sum())

le_target = LabelEncoder()
data["Purchased_Encoded"] = le_target.fit_transform(data["Purchased"])
print("Purchased mapping:", dict(zip(le_target.classes_, le_target.transform(le_target.classes_))))

data["Country_Label"] = LabelEncoder().fit_transform(data["Country"])
print("\nNote: Country_Label implies an artificial order (Brazil < India < Japan),")
print("so LabelEncoder is unsuitable for nominal features in linear models.")
print(data[["Country", "Country_Label", "Purchased", "Purchased_Encoded"]].head())

onehot = OneHotEncoder(sparse_output=False, dtype=int)
encoded = onehot.fit_transform(data[["Country"]])
ohe_df = pd.DataFrame(encoded, columns=onehot.get_feature_names_out(["Country"]), index=data.index)

model_df = pd.concat([data.drop(columns=["Country", "Country_Label"]), ohe_df], axis=1)
print(model_df.head(9))

print(pd.get_dummies(data["Country"], prefix="Country", dtype=int).head())

,Country,Age,Salary,Purchased
0,India,26.0,54000.0,Yes
1,Brazil,25.0,46400.0,No
2,India,51.0,66400.0,Yes
3,Brazil,39.0,NaN,Yes
4,India,43.0,68100.0,Yes
5,Japan,43.0,62000.0,Yes
6,India,48.0,71500.0,Yes
7,Japan,22.0,51300.0,Yes
8,Brazil,NaN,59000.0,No


Country      0
Age          6
Salary       6
Purchased    0
dtype: int64
Learned means (Age, Salary): [4.167000e+01 6.433333e+04]

Missing values after imputation:
Country      0
Age          0
Salary       0
Purchased    0
dtype: int64
Purchased mapping: {'No': np.int64(0), 'Yes': np.int64(1)}

Note: Country_Label implies an artificial order (Brazil < India < Japan),
so LabelEncoder is unsuitable for nominal features in linear models.

,Country,Country_Label,Purchased,Purchased_Encoded
0,India,1,Yes,1
1,Brazil,0,No,0
2,India,1,Yes,1
3,Brazil,0,Yes,1
4,India,1,Yes,1


,Age,Salary,Purchased,Purchased_Encoded,Country_Brazil,Country_India,Country_Japan
0,26.000000,54000.000000,Yes,1,0,1,0
1,25.000000,46400.000000,No,0,1,0,0
2,51.000000,66400.000000,Yes,1,0,1,0
3,39.000000,64333.333333,Yes,1,1,0,0
4,43.000000,68100.000000,Yes,1,0,1,0
5,43.000000,62000.000000,Yes,1,0,0,1
6,48.000000,71500.000000,Yes,1,0,1,0
7,22.000000,51300.000000,Yes,1,0,0,1
8,41.666667,59000.000000,No,0,1,0,0


,Country_Brazil,Country_India,Country_Japan
0,0,1,0
1,1,0,0
2,0,1,0
3,1,0,0
4,0,1,0
